In [2]:
# Install the dataset library
%pip install ucimlrepo pandas

Note: you may need to restart the kernel to use updated packages.


In [3]:
# Phase 1: collect and prepare real sensor data

# Fetch the real dataset
from ucimlrepo import fetch_ucirepo

# Fetch the public sensor dataset from UCI
dataset = fetch_ucirepo(id=799)

# Separate sensor features and target, if provided
X = dataset.data.features
y = dataset.data.targets

print("Dataset downloaded successfully!")
print("Number of rows:", len(X))
print("Number of columns:", len(X.columns))
print("\nColumn names:")
print(X.columns.tolist())

print("\nFirst five records:")
display(X.head())

Dataset downloaded successfully!
Number of rows: 416153
Number of columns: 10

Column names:
['timestamp', 'temperature', 'humidity', 'CO2CosIRValue', 'CO2MG811Value', 'MOX1', 'MOX2', 'MOX3', 'MOX4', 'COValue']

First five records:


,timestamp,temperature,humidity,CO2CosIRValue,CO2MG811Value,MOX1,MOX2,MOX3,MOX4,COValue
0,2019-11-06 11:37:13.038174+01:00,19.48,54.86,128.0,563.0,476.0,731.0,649.0,565.0,128.0
1,2019-11-06 11:37:32.744996+01:00,19.59,54.23,129.0,563.0,477.0,731.0,649.0,565.0,125.0
2,2019-11-06 11:37:53.018087+01:00,19.63,54.05,128.0,566.0,478.0,732.0,649.0,565.0,125.0
3,2019-11-06 11:38:13.093151+01:00,19.64,53.74,128.0,566.0,478.0,732.0,649.0,565.0,125.0
4,2019-11-06 11:38:33.032832+01:00,19.67,53.53,128.0,569.0,480.0,732.0,650.0,565.0,125.0


In [4]:
# Inspect the data
print("Dataset information:")
X.info()

print("\nMissing values per column:")
display(X.isna().sum().sort_values(ascending=False))

print("\nFirst five rows and all columns:")
display(X.head().T)

Dataset information:
<class 'pandas.DataFrame'>
RangeIndex: 416153 entries, 0 to 416152
Data columns (total 10 columns):
 #   Column         Non-Null Count   Dtype  
---  ------         --------------   -----  
 0   timestamp      416153 non-null  str    
 1   temperature    416153 non-null  float64
 2   humidity       416153 non-null  float64
 3   CO2CosIRValue  416153 non-null  float64
 4   CO2MG811Value  416153 non-null  float64
 5   MOX1           416153 non-null  float64
 6   MOX2           416153 non-null  float64
 7   MOX3           416153 non-null  float64
 8   MOX4           416153 non-null  float64
 9   COValue        416153 non-null  float64
dtypes: float64(9), str(1)
memory usage: 44.5 MB

Missing values per column:


timestamp        0
temperature      0
humidity         0
CO2CosIRValue    0
CO2MG811Value    0
MOX1             0
MOX2             0
MOX3             0
MOX4             0
COValue          0
dtype: int64


First five rows and all columns:


,0,1,2,3,4
timestamp,2019-11-06 11:37:13.038174+01:00,2019-11-06 11:37:32.744996+01:00,2019-11-06 11:37:53.018087+01:00,2019-11-06 11:38:13.093151+01:00,2019-11-06 11:38:33.032832+01:00
temperature,19.48,19.59,19.63,19.64,19.67
humidity,54.86,54.23,54.05,53.74,53.53
CO2CosIRValue,128.0,129.0,128.0,128.0,128.0
CO2MG811Value,563.0,563.0,566.0,566.0,569.0
MOX1,476.0,477.0,478.0,478.0,480.0
MOX2,731.0,731.0,732.0,732.0,732.0
MOX3,649.0,649.0,649.0,649.0,650.0
MOX4,565.0,565.0,565.0,565.0,565.0
COValue,128.0,125.0,125.0,125.0,125.0


In [5]:
# Save a local copy
from pathlib import Path

data_dir = Path("iot-platform-v2-data")
data_dir.mkdir(exist_ok=True)

X.to_csv(
    data_dir / "sensor_readings_source.csv",
    index=False
)

if y is not None and len(y.columns) > 0:
    y.to_csv(
        data_dir / "sensor_targets_source.csv",
        index=False
    )

print("Saved source data to:", data_dir.resolve())
# This creates a local folder containing the downloaded sensor data. We are not uploading anything to AWS yet.

Saved source data to: C:\Users\msi\Desktop\Data Analyst\Data Engineer project\iot-platform-v2-data


In [6]:
# Profile and validate the sensor data
# Run the data-quality assessment

import pandas as pd

# Work on a copy; preserve the downloaded source data
df = X.copy()

print("=" * 55)
print("SENSOR DATA QUALITY REPORT")
print("=" * 55)

# 1. Record counts
print("\n1. RECORD COUNTS")
print("Total records:", len(df))
print("Total columns:", len(df.columns))
print("Completely duplicated rows:", df.duplicated().sum())

# 2. Parse timestamps into timezone-aware UTC timestamps
df["timestamp"] = pd.to_datetime(
    df["timestamp"],
    errors="coerce",
    utc=True
)

invalid_timestamps = df["timestamp"].isna().sum()

print("\n2. TIMESTAMP QUALITY")
print("Invalid timestamps:", invalid_timestamps)
print("Earliest timestamp:", df["timestamp"].min())
print("Latest timestamp:", df["timestamp"].max())
print("Duplicate timestamps:", df["timestamp"].duplicated().sum())
print("Chronologically sorted:", df["timestamp"].is_monotonic_increasing)

# 3. Measurement statistics
measurement_cols = [
    "temperature",
    "humidity",
    "CO2CosIRValue",
    "CO2MG811Value",
    "MOX1",
    "MOX2",
    "MOX3",
    "MOX4",
    "COValue"
]

print("\n3. MEASUREMENT STATISTICS")
display(df[measurement_cols].describe().T)

# 4. Check basic temperature and humidity plausibility.
# These are screening thresholds, not definitive sensor specifications.
temperature_flag = (
    (df["temperature"] < -40) |
    (df["temperature"] > 85)
)

humidity_flag = (
    (df["humidity"] < 0) |
    (df["humidity"] > 100)
)

print("\n4. PLAUSIBILITY SCREENING")
print("Temperature outside -40 to 85 °C:", temperature_flag.sum())
print("Humidity outside 0 to 100%:", humidity_flag.sum())

# 5. Time between consecutive observations
sorted_times = df["timestamp"].dropna().sort_values()
intervals = sorted_times.diff().dt.total_seconds().dropna()

print("\n5. READING INTERVALS")
print("Median interval (seconds):", intervals.median())
print("Minimum interval (seconds):", intervals.min())
print("Maximum interval (seconds):", intervals.max())
print("Intervals over 60 seconds:", (intervals > 60).sum())

print("\nQuality assessment complete.")


SENSOR DATA QUALITY REPORT

1. RECORD COUNTS
Total records: 416153
Total columns: 10
Completely duplicated rows: 0

2. TIMESTAMP QUALITY
Invalid timestamps: 0
Earliest timestamp: 2019-11-06 10:37:13.038174+00:00
Latest timestamp: 2020-02-13 10:58:14.014336+00:00
Duplicate timestamps: 0
Chronologically sorted: True

3. MEASUREMENT STATISTICS


,count,mean,std,min,25%,50%,75%,max
temperature,416153.0,20.115751,2.295836,13.53,18.64,20.10,21.63,26.14
humidity,416153.0,52.236446,4.343208,38.16,49.05,52.31,55.64,69.26
CO2CosIRValue,416153.0,102.459415,39.026812,21.00,76.00,105.00,117.00,591.00
CO2MG811Value,416153.0,539.086523,34.934361,408.00,510.00,528.00,563.00,632.00
MOX1,416153.0,500.289413,49.795030,262.00,468.00,502.00,536.00,626.00
MOX2,416153.0,718.708761,31.296907,514.00,703.00,719.00,738.00,808.00
MOX3,416153.0,672.788842,36.354293,447.00,655.00,674.00,693.00,777.00
MOX4,416153.0,609.446752,42.703923,355.00,587.00,611.00,635.00,727.00
COValue,416153.0,120.124310,35.992508,74.00,103.00,113.00,128.00,918.00



4. PLAUSIBILITY SCREENING
Temperature outside -40 to 85 °C: 0
Humidity outside 0 to 100%: 0

5. READING INTERVALS
Median interval (seconds): 20.004286999999998
Minimum interval (seconds): 1.757329
Maximum interval (seconds): 166398.560293
Intervals over 60 seconds: 31

Quality assessment complete.


In [7]:
%pip install pyarrow

Note: you may need to restart the kernel to use updated packages.


In [8]:
# Create the standardized dataset

import pandas as pd
import json
from pathlib import Path

# Project configuration
PROJECT_DIR = Path("iot-platform-v2-data")
PROJECT_DIR.mkdir(exist_ok=True)

SOURCE_DATASET = "UCI_Single_Elder_Home_Monitoring"
SOURCE_DATASET_ID = 799

# Preserve the original downloaded file if it already exists.
source_path = PROJECT_DIR / "sensor_readings_source.csv"

if not source_path.exists():
    X.to_csv(source_path, index=False)

# Work on a separate DataFrame.
source_df = X.copy()

# Standardize timestamps to UTC.
source_df["timestamp"] = pd.to_datetime(
    source_df["timestamp"],
    errors="coerce",
    utc=True
)

# Numeric sensor fields from the source dataset.
measurement_cols = [
    "temperature",
    "humidity",
    "CO2CosIRValue",
    "CO2MG811Value",
    "MOX1",
    "MOX2",
    "MOX3",
    "MOX4",
    "COValue"
]

for col in measurement_cols:
    source_df[col] = pd.to_numeric(
        source_df[col],
        errors="coerce"
    )

# Create a standardized table.
# These record IDs identify rows in our pipeline, not physical devices.
processed_df = source_df.copy()

processed_df.insert(
    0,
    "source_record_id",
    range(1, len(processed_df) + 1)
)

processed_df["source_dataset"] = SOURCE_DATASET
processed_df["source_dataset_id"] = SOURCE_DATASET_ID

# Identify records with invalid timestamps or missing/non-finite
# numeric measurements. Do not silently discard them.
import numpy as np

valid_timestamp = processed_df["timestamp"].notna()

valid_measurements = pd.Series(
    np.isfinite(
        processed_df[measurement_cols].to_numpy(dtype=float)
    ).all(axis=1),
    index=processed_df.index
)

valid_mask = valid_timestamp & valid_measurements

clean_df = processed_df.loc[valid_mask].copy()
quarantine_df = processed_df.loc[~valid_mask].copy()

# Record the UTC-normalized event time.
clean_df = clean_df.rename(
    columns={"timestamp": "event_time"}
)

# Save the pipeline outputs.
clean_path = PROJECT_DIR / "sensor_readings_clean.parquet"
quarantine_path = PROJECT_DIR / "sensor_readings_quarantine.csv"
quality_path = PROJECT_DIR / "data_quality_report.json"

clean_df.to_parquet(clean_path, index=False)
quarantine_df.to_csv(quarantine_path, index=False)

# Check unusually large gaps between readings.
ordered_times = clean_df["event_time"].sort_values()
intervals = ordered_times.diff().dt.total_seconds()

large_gaps = intervals[intervals > 60]

quality_report = {
    "source_dataset": SOURCE_DATASET,
    "source_dataset_id": SOURCE_DATASET_ID,
    "input_records": int(len(processed_df)),
    "clean_records": int(len(clean_df)),
    "quarantined_records": int(len(quarantine_df)),
    "count_reconciles": (
        len(processed_df) == len(clean_df) + len(quarantine_df)
    ),
    "invalid_timestamp_records": int((~valid_timestamp).sum()),
    "invalid_measurement_records": int((~valid_measurements).sum()),
    "intervals_over_60_seconds": int(len(large_gaps)),
    "maximum_observed_gap_seconds": float(intervals.max()),
    "timestamp_standard": "UTC",
    "physical_device_id_available_in_source": False
}

with open(quality_path, "w", encoding="utf-8") as f:
    json.dump(quality_report, f, indent=2)

print("INGESTION PREPARATION COMPLETE")
print("--------------------------------")
print("Input records:", len(processed_df))
print("Clean records:", len(clean_df))
print("Quarantined records:", len(quarantine_df))
print("Count reconciliation:", quality_report["count_reconciles"])
print("Clean data file:", clean_path)
print("Quality report:", quality_path)
print("\nClean schema:")
print(clean_df.dtypes)


INGESTION PREPARATION COMPLETE
--------------------------------
Input records: 416153
Clean records: 416153
Quarantined records: 0
Count reconciliation: True
Clean data file: iot-platform-v2-data\sensor_readings_clean.parquet
Quality report: iot-platform-v2-data\data_quality_report.json

Clean schema:
source_record_id                   int64
event_time           datetime64[us, UTC]
temperature                      float64
humidity                         float64
CO2CosIRValue                    float64
CO2MG811Value                    float64
MOX1                             float64
MOX2                             float64
MOX3                             float64
MOX4                             float64
COValue                          float64
source_dataset                       str
source_dataset_id                  int64
dtype: object


In [1]:
# Verify the result
import pandas as pd
import json

check_df = pd.read_parquet(
    "iot-platform-v2-data/sensor_readings_clean.parquet"
)

with open(
    "iot-platform-v2-data/data_quality_report.json",
    encoding="utf-8"
) as f:
    report = json.load(f)

print("Clean data shape:", check_df.shape)
print("First five records:")
display(check_df.head())

print("\nQuality report:")
print(json.dumps(report, indent=2))

print("\nParquet file size (MB):",
      round(
          __import__("os").path.getsize(
              "iot-platform-v2-data/sensor_readings_clean.parquet"
          ) / (1024 * 1024),
          2
      ))

Clean data shape: (416153, 13)
First five records:


,source_record_id,event_time,temperature,humidity,CO2CosIRValue,CO2MG811Value,MOX1,MOX2,MOX3,MOX4,COValue,source_dataset,source_dataset_id
0,1,2019-11-06 10:37:13.038174+00:00,19.48,54.86,128.0,563.0,476.0,731.0,649.0,565.0,128.0,UCI_Single_Elder_Home_Monitoring,799
1,2,2019-11-06 10:37:32.744996+00:00,19.59,54.23,129.0,563.0,477.0,731.0,649.0,565.0,125.0,UCI_Single_Elder_Home_Monitoring,799
2,3,2019-11-06 10:37:53.018087+00:00,19.63,54.05,128.0,566.0,478.0,732.0,649.0,565.0,125.0,UCI_Single_Elder_Home_Monitoring,799
3,4,2019-11-06 10:38:13.093151+00:00,19.64,53.74,128.0,566.0,478.0,732.0,649.0,565.0,125.0,UCI_Single_Elder_Home_Monitoring,799
4,5,2019-11-06 10:38:33.032832+00:00,19.67,53.53,128.0,569.0,480.0,732.0,650.0,565.0,125.0,UCI_Single_Elder_Home_Monitoring,799



Quality report:
{
  "source_dataset": "UCI_Single_Elder_Home_Monitoring",
  "source_dataset_id": 799,
  "input_records": 416153,
  "clean_records": 416153,
  "quarantined_records": 0,
  "count_reconciles": true,
  "invalid_timestamp_records": 0,
  "invalid_measurement_records": 0,
  "intervals_over_60_seconds": 31,
  "maximum_observed_gap_seconds": 166398.560293,
  "timestamp_standard": "UTC",
  "physical_device_id_available_in_source": false
}

Parquet file size (MB): 7.3


In [5]:
# Phase 2 — Prepare the data for AWS S3

# Upload the files from Jupyter
import boto3
from pathlib import Path

# Existing AWS bucket and new project area
BUCKET = "amogh-iot-data-platform-2026-946486414750-eu-north-1-an"
PREFIX = "iot-platform-v2"

# Use credentials from your local AWS profile
session = boto3.Session(
    profile_name="iot-project",
    region_name="eu-north-1"
)

s3 = session.client("s3")

# Files created in Phase 1
project_dir = Path("iot-platform-v2-data")

files_to_upload = {
    project_dir / "sensor_readings_source.csv":
        f"{PREFIX}/raw/sensor_readings_source.csv",

    project_dir / "sensor_readings_clean.parquet":
        f"{PREFIX}/processed/sensor_readings_clean.parquet",

    project_dir / "data_quality_report.json":
        f"{PREFIX}/processed/data_quality_report.json",

    project_dir / "sensor_readings_quarantine.csv":
        f"{PREFIX}/quarantine/sensor_readings_quarantine.csv",
}

for local_path, s3_key in files_to_upload.items():
    if not local_path.exists():
        raise FileNotFoundError(f"File not found: {local_path}")

    s3.upload_file(str(local_path), BUCKET, s3_key)
    print(f"Uploaded: {local_path.name} -> s3://{BUCKET}/{s3_key}")

print("\nUpload step completed.")

Uploaded: sensor_readings_source.csv -> s3://amogh-iot-data-platform-2026-946486414750-eu-north-1-an/iot-platform-v2/raw/sensor_readings_source.csv
Uploaded: sensor_readings_clean.parquet -> s3://amogh-iot-data-platform-2026-946486414750-eu-north-1-an/iot-platform-v2/processed/sensor_readings_clean.parquet
Uploaded: data_quality_report.json -> s3://amogh-iot-data-platform-2026-946486414750-eu-north-1-an/iot-platform-v2/processed/data_quality_report.json
Uploaded: sensor_readings_quarantine.csv -> s3://amogh-iot-data-platform-2026-946486414750-eu-north-1-an/iot-platform-v2/quarantine/sensor_readings_quarantine.csv

Upload step completed.


In [6]:
# Verify the upload
response = s3.list_objects_v2(
    Bucket=BUCKET,
    Prefix="iot-platform-v2/"
)

for obj in response.get("Contents", []):
    print(f"{obj['Key']}  |  {obj['Size']:,} bytes")

iot-platform-v2/processed/  |  0 bytes
iot-platform-v2/processed/data_quality_report.json  |  445 bytes
iot-platform-v2/processed/sensor_readings_clean.parquet  |  7,651,253 bytes
iot-platform-v2/quarantine/  |  0 bytes
iot-platform-v2/quarantine/sensor_readings_quarantine.csv  |  138 bytes
iot-platform-v2/raw/  |  0 bytes
iot-platform-v2/raw/sensor_readings_source.csv  |  36,287,827 bytes
iot-platform-v2/warehouse/iceberg/  |  0 bytes


In [7]:
# Phase 3 — Check Spark before connecting it to S3

import pyspark

print("PySpark version:", pyspark.__version__)

try:
    print("Spark version:", spark.version)

    hadoop_version = (
        spark.sparkContext._jvm
        .org.apache.hadoop.util.VersionInfo.getVersion()
    )
    print("Hadoop version:", hadoop_version)

    hadoop_conf = spark.sparkContext._jsc.hadoopConfiguration()
    print("S3A filesystem:", hadoop_conf.get("fs.s3a.impl"))

except NameError:
    print("Spark session is not defined. We need to create it first.")

PySpark version: 4.2.0
Spark session is not defined. We need to create it first.


In [8]:
# Start Spark because Spark session is not defined. We need to create it first.
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("IoT-Platform-V2")
    .master("local[*]")
    .getOrCreate()
)

print("Spark session created successfully!")
print("Spark version:", spark.version)

hadoop_version = (
    spark.sparkContext._jvm
    .org.apache.hadoop.util.VersionInfo.getVersion()
)
print("Hadoop version:", hadoop_version)

hadoop_conf = spark.sparkContext._jsc.hadoopConfiguration()
print("S3A filesystem:", hadoop_conf.get("fs.s3a.impl"))

C:\Users\msi\.conda\envs\pyspark\Lib\site-packages\pyspark\testing\utils.py:127: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()


Spark session created successfully!
Spark version: 4.2.0
Hadoop version: 3.5.0
S3A filesystem: org.apache.hadoop.fs.s3a.S3AFileSystem


In [9]:
# Check whether the S3 connector JAR is loaded
jvm = spark.sparkContext._jvm

classes_to_check = [
    "org.apache.hadoop.fs.s3a.S3AFileSystem",
    "org.apache.hadoop.fs.s3a.S3AFileSystem",
    "software.amazon.awssdk.services.s3.S3Client",
]

for class_name in classes_to_check:
    try:
        jvm.java.lang.Class.forName(class_name)
        print("FOUND:", class_name)
    except Exception as e:
        print("NOT FOUND:", class_name)

NOT FOUND: org.apache.hadoop.fs.s3a.S3AFileSystem
NOT FOUND: org.apache.hadoop.fs.s3a.S3AFileSystem
NOT FOUND: software.amazon.awssdk.services.s3.S3Client


In [1]:
# Restart Spark with the correct connector
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("IoT-Platform-V2")
    .master("local[*]")
    .config(
        "spark.jars.packages",
        "org.apache.hadoop:hadoop-aws:3.5.0"
    )
    .getOrCreate()
)

print("Spark:", spark.version)

print(
    "Hadoop:",
    spark.sparkContext._jvm
    .org.apache.hadoop.util.VersionInfo.getVersion()
)

C:\Users\msi\.conda\envs\pyspark\Lib\site-packages\pyspark\testing\utils.py:127: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()


Py4JJavaError: An error occurred while calling None.org.apache.spark.api.java.JavaSparkContext.
: java.lang.RuntimeException: java.io.FileNotFoundException: java.io.FileNotFoundException: HADOOP_HOME and hadoop.home.dir are unset. -see https://cwiki.apache.org/confluence/display/HADOOP2/WindowsProblems
	at org.apache.hadoop.util.Shell.getWinUtilsPath(Shell.java:790)
	at org.apache.hadoop.util.Shell.getSetPermissionCommand(Shell.java:299)
	at org.apache.hadoop.fs.FileUtil.chmod(FileUtil.java:1305)
	at org.apache.hadoop.fs.FileUtil.chmod(FileUtil.java:1291)
	at org.apache.spark.util.Utils$.fetchFile(Utils.scala:432)
	at org.apache.spark.SparkContext.addFile(SparkContext.scala:1875)
	at org.apache.spark.SparkContext.$anonfun$new$17(SparkContext.scala:551)
	at org.apache.spark.SparkContext.$anonfun$new$17$adapted(SparkContext.scala:551)
	at scala.collection.immutable.List.foreach(List.scala:323)
	at org.apache.spark.SparkContext.<init>(SparkContext.scala:551)
	at org.apache.spar

In [2]:
# Spark is failing before it can start because Windows cannot find Hadoop’s Windows utility, winutils.exe
# so Check your Windows Hadoop setup
import os
from pathlib import Path

print("HADOOP_HOME:", os.environ.get("HADOOP_HOME"))
print("JAVA_HOME:", os.environ.get("JAVA_HOME"))

for path in [
    Path(r"C:\hadoop"),
    Path(r"C:\hadoop\bin\winutils.exe"),
    Path(r"C:\Program Files\Eclipse Adoptium\jdk-17.0.20.101-hotspot"),
]:
    print(path, "->", "EXISTS" if path.exists() else "NOT FOUND")

HADOOP_HOME: None
JAVA_HOME: C:\Program Files\Eclipse Adoptium\jdk-17.0.20.101-hotspot\
C:\hadoop -> NOT FOUND
C:\hadoop\bin\winutils.exe -> NOT FOUND
C:\Program Files\Eclipse Adoptium\jdk-17.0.20.101-hotspot -> EXISTS


In [1]:
# Java is installed correctly, but Hadoop's Windows utilities are missing. Let's fix this without downloading anything yet.
# Check whether Spark needs winutils.exe in your setup
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("IoT-Platform-V2")
    .master("local[*]")
    .config(
        "spark.jars.packages",
        "org.apache.hadoop:hadoop-aws:3.5.0"
    )
    .config("spark.files.useFetchCache", "false")
    .getOrCreate()
)

print("Spark started:", spark.version)

C:\Users\msi\.conda\envs\pyspark\Lib\site-packages\pyspark\testing\utils.py:127: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()


Py4JJavaError: An error occurred while calling None.org.apache.spark.api.java.JavaSparkContext.
: java.lang.RuntimeException: java.io.FileNotFoundException: java.io.FileNotFoundException: HADOOP_HOME and hadoop.home.dir are unset. -see https://cwiki.apache.org/confluence/display/HADOOP2/WindowsProblems
	at org.apache.hadoop.util.Shell.getWinUtilsPath(Shell.java:790)
	at org.apache.hadoop.util.Shell.getSetPermissionCommand(Shell.java:299)
	at org.apache.hadoop.fs.FileUtil.chmod(FileUtil.java:1305)
	at org.apache.hadoop.fs.FileUtil.chmod(FileUtil.java:1291)
	at org.apache.spark.util.Utils$.fetchFile(Utils.scala:432)
	at org.apache.spark.SparkContext.addFile(SparkContext.scala:1875)
	at org.apache.spark.SparkContext.$anonfun$new$17(SparkContext.scala:551)
	at org.apache.spark.SparkContext.$anonfun$new$17$adapted(SparkContext.scala:551)
	at scala.collection.immutable.List.foreach(List.scala:323)
	at org.apache.spark.SparkContext.<init>(SparkContext.scala:551)
	at org.apache.spar

In [ ]:
# The error confirms that adding the S3 connector did not get Spark past the Windows startup problem. 
# The important part is still: HADOOP_HOME and hadoop.home.dir are unset
# using WSL2 (Linux on Windows) for Spark? WSL2 avoids the Windows winutils.exe issue and is a good option for a data-engineering project
